# 02 Build Gold

Builds the **gold** schema from the silver tables: a star schema with friendly column names and **current dates**, ready for a Direct Lake semantic model.

The source data runs from late 2010 to mid 2014. This notebook moves every date forward by a whole number of days so the data always looks current:

| Data | Rule |
| --- | --- |
| Internet sales | The latest order date becomes today |
| Reseller sales and sales quotas | The latest reseller order date becomes today (quotas move with the sales they measure) |
| Finance, call center, currency rates, inventory, survey | The latest date in each table becomes today |
| Calendar | Generated: whole calendar years covering every date above |

Run it daily (the `AdventureWorks Daily Refresh` pipeline does this) and the data rolls forward every day. Direct Lake picks up the new data on its own.

In [ ]:
# Parameters. A pipeline or the setup notebook can override these.
LAKEHOUSE_NAME = "AdventureWorksLH"
AS_OF_DATE = ""   # leave empty for today (UTC), or set a date such as "2026-12-31" to freeze the demo on that day

In [ ]:
from datetime import date

import notebookutils
from pyspark.sql import functions as F

lakehouse = notebookutils.lakehouse.get(LAKEHOUSE_NAME)
ROOT = f"abfss://{lakehouse.workspaceId}@onelake.dfs.fabric.microsoft.com/{lakehouse.id}"
SILVER = f"{ROOT}/Tables/silver"
GOLD = f"{ROOT}/Tables/gold"

# V-Order and optimized write make the tables faster for Direct Lake and the SQL analytics endpoint.
spark.conf.set("spark.sql.parquet.vorder.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")

TODAY = date.fromisoformat(AS_OF_DATE) if AS_OF_DATE else spark.sql("SELECT current_date()").first()[0]
print(f"Building gold as of {TODAY}")


def ensure_schema(schema):
    """Create the schema in the Lakehouse if it is not there yet."""
    workspace = notebookutils.runtime.context["currentWorkspaceName"]
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{workspace}`.`{LAKEHOUSE_NAME}`.`{schema}`")
    except Exception as error:
        # Writing a table under Tables/<schema> also creates the schema, so carry on.
        print(f"Could not create schema {schema} with Spark SQL ({type(error).__name__}); it will be created by the first write.")


def silver(name):
    return spark.read.format("delta").load(f"{SILVER}/{name}")


gold = {}   # every gold table built in this run, for the checks at the end


def save_gold(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(f"{GOLD}/{name}")
    gold[name] = spark.read.format("delta").load(f"{GOLD}/{name}")
    print(f"Built gold.{name}")


ensure_schema("gold")

# Make every silver table available to Spark SQL under its own name.
SILVER_TABLES = [
    "DimAccount", "DimCurrency", "DimCustomer", "DimDepartmentGroup", "DimEmployee", "DimGeography",
    "DimOrganization", "DimProduct", "DimProductCategory", "DimProductSubcategory", "DimPromotion",
    "DimReseller", "DimSalesTerritory", "DimScenario", "FactCallCenter", "FactCurrencyRate", "FactFinance",
    "FactInternetSales", "FactProductInventory", "FactResellerSales", "FactSalesQuota", "FactSurveyResponse",
]
for name in SILVER_TABLES:
    silver(name).createOrReplaceTempView(name)

## How far to move each set of dates

In [ ]:
def days_to_today(table, column):
    """Number of days between the latest date in a silver table and today."""
    latest = silver(table).agg(F.max(column)).first()[0]
    return (TODAY - latest).days


SHIFT = {
    "internet": days_to_today("FactInternetSales", "OrderDate"),
    "reseller": days_to_today("FactResellerSales", "OrderDate"),
    "finance": days_to_today("FactFinance", "Date"),
    "callcenter": days_to_today("FactCallCenter", "Date"),
    "currency": days_to_today("FactCurrencyRate", "Date"),
    "inventory": days_to_today("FactProductInventory", "MovementDate"),
    "survey": days_to_today("FactSurveyResponse", "Date"),
}
for name, days in SHIFT.items():
    print(f"{name:<11} moves forward {days} days")

## Dimensions

Product and geography hierarchies are flattened into their dimension, single-letter codes become words, and columns get friendly names.

In [ ]:
save_gold(spark.sql("""
    SELECT p.ProductKey,
           p.ProductAlternateKey,
           p.EnglishProductName             AS ProductName,
           s.EnglishProductSubcategoryName  AS ProductSubcategory,
           c.EnglishProductCategoryName     AS ProductCategory,
           p.ModelName,
           p.Color,
           p.Size,
           p.SizeRange,
           CASE p.ProductLine WHEN 'R' THEN 'Road' WHEN 'M' THEN 'Mountain' WHEN 'T' THEN 'Touring' WHEN 'S' THEN 'Standard' END AS ProductLine,
           CASE p.Class WHEN 'H' THEN 'High' WHEN 'M' THEN 'Medium' WHEN 'L' THEN 'Low' END AS Class,
           CASE p.Style WHEN 'W' THEN 'Womens' WHEN 'M' THEN 'Mens' WHEN 'U' THEN 'Universal' END AS Style,
           p.StandardCost,
           p.ListPrice,
           p.DealerPrice,
           p.Weight,
           p.DaysToManufacture,
           p.FinishedGoodsFlag              AS IsFinishedGood,
           p.EnglishDescription             AS Description,
           COALESCE(p.Status = 'Current', false) AS IsCurrent
    FROM DimProduct p
    LEFT JOIN DimProductSubcategory s ON p.ProductSubcategoryKey = s.ProductSubcategoryKey
    LEFT JOIN DimProductCategory c    ON s.ProductCategoryKey = c.ProductCategoryKey
"""), "DimProduct")

save_gold(spark.sql(f"""
    SELECT c.CustomerKey,
           c.CustomerAlternateKey,
           concat_ws(' ', c.FirstName, c.MiddleName, c.LastName) AS CustomerName,
           date_add(c.BirthDate, {SHIFT['internet']})            AS BirthDate,
           CASE c.MaritalStatus WHEN 'M' THEN 'Married' WHEN 'S' THEN 'Single' END AS MaritalStatus,
           CASE c.Gender WHEN 'M' THEN 'Male' WHEN 'F' THEN 'Female' END           AS Gender,
           c.YearlyIncome,
           c.TotalChildren,
           c.NumberChildrenAtHome,
           c.EnglishEducation                                    AS Education,
           c.EnglishOccupation                                   AS Occupation,
           c.HouseOwnerFlag = '1'                                AS IsHomeOwner,
           c.NumberCarsOwned,
           c.CommuteDistance,
           date_add(c.DateFirstPurchase, {SHIFT['internet']})    AS DateFirstPurchase,
           g.City,
           g.StateProvinceName                                   AS StateProvince,
           g.EnglishCountryRegionName                            AS Country,
           g.PostalCode
    FROM DimCustomer c
    LEFT JOIN DimGeography g ON c.GeographyKey = g.GeographyKey
"""), "DimCustomer")

save_gold(spark.sql("""
    SELECT r.ResellerKey,
           r.ResellerAlternateKey,
           r.ResellerName,
           r.BusinessType,
           r.NumberEmployees,
           r.ProductLine,
           r.AnnualSales,
           r.AnnualRevenue,
           r.BankName,
           r.YearOpened,
           g.City,
           g.StateProvinceName        AS StateProvince,
           g.EnglishCountryRegionName AS Country,
           g.PostalCode
    FROM DimReseller r
    LEFT JOIN DimGeography g ON r.GeographyKey = g.GeographyKey
"""), "DimReseller")

save_gold(spark.sql(f"""
    SELECT EmployeeKey,
           ParentEmployeeKey,
           concat_ws(' ', FirstName, MiddleName, LastName) AS EmployeeName,
           Title,
           DepartmentName                                  AS Department,
           date_add(HireDate, {SHIFT['reseller']})         AS HireDate,
           SalesPersonFlag                                 AS IsSalesPerson,
           CurrentFlag                                     AS IsCurrent,
           SalesTerritoryKey
    FROM DimEmployee
"""), "DimEmployee")

save_gold(spark.sql("""
    SELECT PromotionKey,
           EnglishPromotionName     AS PromotionName,
           EnglishPromotionType     AS PromotionType,
           EnglishPromotionCategory AS PromotionCategory,
           DiscountPct,
           MinQty,
           MaxQty
    FROM DimPromotion
"""), "DimPromotion")

save_gold(spark.sql("""
    SELECT SalesTerritoryKey,
           SalesTerritoryRegion  AS Region,
           SalesTerritoryCountry AS Country,
           SalesTerritoryGroup   AS TerritoryGroup
    FROM DimSalesTerritory
"""), "DimSalesTerritory")

save_gold(spark.sql("""
    SELECT CurrencyKey,
           CurrencyAlternateKey AS CurrencyCode,
           CurrencyName
    FROM DimCurrency
"""), "DimCurrency")

# The finance dimensions have no dates or codes to tidy, so they are carried over as they are.
for name in ["DimAccount", "DimOrganization", "DimDepartmentGroup", "DimScenario"]:
    save_gold(silver(name), name)

## Facts

Internet and reseller sales become one `FactSales` table with a `Channel` column, so a model needs one sales table and one set of measures.

In [ ]:
SALES_COLUMNS = """
           SalesOrderNumber,
           SalesOrderLineNumber,
           ProductKey,
           {customer}  AS CustomerKey,
           {reseller}  AS ResellerKey,
           {employee}  AS EmployeeKey,
           PromotionKey,
           CurrencyKey,
           SalesTerritoryKey,
           date_add(OrderDate, {days}) AS OrderDate,
           date_add(DueDate, {days})   AS DueDate,
           date_add(ShipDate, {days})  AS ShipDate,
           OrderQuantity,
           UnitPrice,
           UnitPriceDiscountPct,
           DiscountAmount,
           ProductStandardCost,
           TotalProductCost,
           SalesAmount,
           TaxAmt,
           Freight
"""

save_gold(spark.sql(
    "SELECT 'Internet' AS Channel,"
    + SALES_COLUMNS.format(customer="CustomerKey", reseller="CAST(NULL AS INT)", employee="CAST(NULL AS INT)", days=SHIFT["internet"])
    + " FROM FactInternetSales UNION ALL SELECT 'Reseller' AS Channel,"
    + SALES_COLUMNS.format(customer="CAST(NULL AS INT)", reseller="ResellerKey", employee="EmployeeKey", days=SHIFT["reseller"])
    + " FROM FactResellerSales"
), "FactSales")

save_gold(spark.sql(f"""
    SELECT SalesQuotaKey,
           EmployeeKey,
           date_add(Date, {SHIFT['reseller']}) AS Date,
           SalesAmountQuota
    FROM FactSalesQuota
"""), "FactSalesQuota")

save_gold(spark.sql(f"""
    SELECT FinanceKey,
           date_add(Date, {SHIFT['finance']}) AS Date,
           OrganizationKey,
           DepartmentGroupKey,
           ScenarioKey,
           AccountKey,
           Amount
    FROM FactFinance
"""), "FactFinance")

save_gold(spark.sql(f"""
    SELECT FactCallCenterID,
           date_add(Date, {SHIFT['callcenter']}) AS Date,
           WageType,
           Shift,
           LevelOneOperators,
           LevelTwoOperators,
           TotalOperators,
           Calls,
           AutomaticResponses,
           Orders,
           IssuesRaised,
           AverageTimePerIssue,
           ServiceGrade
    FROM FactCallCenter
"""), "FactCallCenter")

save_gold(spark.sql(f"""
    SELECT CurrencyKey,
           date_add(Date, {SHIFT['currency']}) AS Date,
           AverageRate,
           EndOfDayRate
    FROM FactCurrencyRate
"""), "FactCurrencyRate")

save_gold(spark.sql(f"""
    SELECT ProductKey,
           date_add(MovementDate, {SHIFT['inventory']}) AS Date,
           UnitCost,
           UnitsIn,
           UnitsOut,
           UnitsBalance
    FROM FactProductInventory
"""), "FactProductInventory")

save_gold(spark.sql(f"""
    SELECT SurveyResponseKey,
           date_add(Date, {SHIFT['survey']}) AS Date,
           CustomerKey,
           ProductCategoryKey,
           EnglishProductCategoryName    AS ProductCategory,
           ProductSubcategoryKey,
           EnglishProductSubcategoryName AS ProductSubcategory
    FROM FactSurveyResponse
"""), "FactSurveyResponse")

## Calendar

`DimDate` is generated, not shifted, so it always holds whole calendar years. The fiscal year starts on July 1, as in AdventureWorks.

In [ ]:
def build_calendar(first_day, last_day):
    """One row per day from first_day to last_day, with calendar and fiscal attributes."""
    days = spark.sql(f"SELECT explode(sequence(DATE'{first_day}', DATE'{last_day}', INTERVAL 1 DAY)) AS Date")
    days.createOrReplaceTempView("calendar_days")
    return spark.sql("""
        SELECT CAST(date_format(Date, 'yyyyMMdd') AS INT)   AS DateKey,
               Date,
               year(Date)                                   AS Year,
               quarter(Date)                                AS Quarter,
               month(Date)                                  AS MonthNumber,
               date_format(Date, 'MMMM')                    AS MonthName,
               date_format(Date, 'yyyy MMM')                AS YearMonth,
               year(Date) * 100 + month(Date)               AS YearMonthSort,
               dayofmonth(Date)                             AS DayOfMonth,
               dayofyear(Date)                              AS DayOfYear,
               dayofweek(Date)                              AS DayOfWeekNumber,   -- 1 = Sunday
               date_format(Date, 'EEEE')                    AS DayOfWeekName,
               -- Week 1 starts on January 1 and a new week starts every Sunday
               CAST(floor((dayofyear(Date) + dayofweek(trunc(Date, 'year')) - 2) / 7) + 1 AS INT) AS WeekOfYear,
               CASE WHEN month(Date) <= 6 THEN 1 ELSE 2 END AS Semester,
               CASE WHEN month(Date) <= 6 THEN year(Date) ELSE year(Date) + 1 END AS FiscalYear,
               CASE WHEN month(Date) <= 3 THEN 3 WHEN month(Date) <= 6 THEN 4 WHEN month(Date) <= 9 THEN 1 ELSE 2 END AS FiscalQuarter,
               CASE WHEN month(Date) <= 6 THEN 2 ELSE 1 END AS FiscalSemester
        FROM calendar_days
    """)


# Every date column in a gold fact table. The calendar must cover all of them.
FACT_DATES = {
    "FactSales": ["OrderDate", "DueDate", "ShipDate"],
    "FactSalesQuota": ["Date"],
    "FactFinance": ["Date"],
    "FactCallCenter": ["Date"],
    "FactCurrencyRate": ["Date"],
    "FactProductInventory": ["Date"],
    "FactSurveyResponse": ["Date"],
}
bounds = [gold[table].agg(F.min(column), F.max(column)).first() for table, columns in FACT_DATES.items() for column in columns]
first_day = date(min(low for low, _ in bounds).year, 1, 1)
last_day = date(max(high for _, high in bounds).year, 12, 31)

save_gold(build_calendar(first_day, last_day), "DimDate")

## Validate

In [ ]:
problems = []

# 1. Nothing was lost or changed on the way from silver to gold.
silver_sales = spark.sql("""
    SELECT 'Internet' AS Channel, count(*) AS n, sum(SalesAmount) AS amount FROM FactInternetSales
    UNION ALL
    SELECT 'Reseller', count(*), sum(SalesAmount) FROM FactResellerSales
""").collect()
gold_sales = {r.Channel: r for r in gold["FactSales"].groupBy("Channel").agg(F.count("*").alias("n"), F.sum("SalesAmount").alias("amount"), F.max("OrderDate").alias("latest")).collect()}
for row in silver_sales:
    built = gold_sales.get(row.Channel)
    if built is None or built.n != row.n or built.amount != row.amount:
        problems.append(f"FactSales {row.Channel}: silver has {row.n} rows / {row.amount}, gold has {built.n if built else 0} rows / {built.amount if built else 0}")
    elif built.latest != TODAY:
        problems.append(f"FactSales {row.Channel}: latest order date is {built.latest}, expected {TODAY}")

for name, silver_name in [("FactSalesQuota", "FactSalesQuota"), ("FactFinance", "FactFinance"), ("FactCallCenter", "FactCallCenter"),
                          ("FactCurrencyRate", "FactCurrencyRate"), ("FactProductInventory", "FactProductInventory"),
                          ("FactSurveyResponse", "FactSurveyResponse"), ("DimProduct", "DimProduct"), ("DimCustomer", "DimCustomer"),
                          ("DimReseller", "DimReseller"), ("DimEmployee", "DimEmployee")]:
    if gold[name].count() != silver(silver_name).count():
        problems.append(f"{name}: row count differs from silver.{silver_name}")

# 2. Keys are unique.
KEYS = {
    "DimDate": ["Date"], "DimProduct": ["ProductKey"], "DimCustomer": ["CustomerKey"], "DimReseller": ["ResellerKey"],
    "DimEmployee": ["EmployeeKey"], "DimPromotion": ["PromotionKey"], "DimSalesTerritory": ["SalesTerritoryKey"],
    "DimCurrency": ["CurrencyKey"], "FactSales": ["SalesOrderNumber", "SalesOrderLineNumber"], "FactSalesQuota": ["SalesQuotaKey"],
}
for name, key in KEYS.items():
    if gold[name].select(*key).distinct().count() != gold[name].count():
        problems.append(f"{name}: key {key} is not unique")

# 3. Every relationship the semantic model uses resolves.
RELATIONSHIPS = [
    ("FactSales", "ProductKey", "DimProduct", "ProductKey"),
    ("FactSales", "CustomerKey", "DimCustomer", "CustomerKey"),
    ("FactSales", "ResellerKey", "DimReseller", "ResellerKey"),
    ("FactSales", "EmployeeKey", "DimEmployee", "EmployeeKey"),
    ("FactSales", "PromotionKey", "DimPromotion", "PromotionKey"),
    ("FactSales", "CurrencyKey", "DimCurrency", "CurrencyKey"),
    ("FactSales", "SalesTerritoryKey", "DimSalesTerritory", "SalesTerritoryKey"),
    ("FactSalesQuota", "EmployeeKey", "DimEmployee", "EmployeeKey"),
] + [(table, column, "DimDate", "Date") for table, columns in FACT_DATES.items() for column in columns]
for child, child_col, parent, parent_col in RELATIONSHIPS:
    orphans = (
        gold[child].select(F.col(child_col).alias("k")).where("k IS NOT NULL").distinct()
        .join(gold[parent].select(F.col(parent_col).alias("k")).distinct(), "k", "left_anti")
        .count()
    )
    if orphans:
        problems.append(f"{child}.{child_col}: {orphans} values not found in {parent}.{parent_col}")

# 4. The calendar has no gaps.
calendar = gold["DimDate"].agg(F.min("Date"), F.max("Date"), F.count("*")).first()
if (calendar[1] - calendar[0]).days + 1 != calendar[2]:
    problems.append("DimDate: the calendar has gaps or duplicate days")

if problems:
    raise Exception("Validation failed:\n" + "\n".join(problems))
print(f"All {len(gold)} gold tables built and validated as of {TODAY}. BEST DAY EVER!!!")